# Classify Italian Brainrot characters

Train a small YOLO classifier on the [Italian Brainrot Images](https://www.kaggle.com/datasets/bubblepw/italian-brainrot-images) dataset, then watch it name a character.

Folder name in dataset = Answer

## Set up your computer

### 1. Make a Kaggle account

Go to [kaggle.com](https://www.kaggle.com) and sign up.

### 2. Open a terminal

- **Windows:** press the Windows key, type `PowerShell`, and open it.
- **Mac:** press Command+Space, type `Terminal`, and open it.
- **Linux:** open the Terminal app.

### 3. Save your Kaggle token

Open [kaggle.com/settings/api](https://www.kaggle.com/settings/api). Click **Generate New Token**. Kaggle shows one command. Paste it into the terminal and run it. That command saves the token.

### 4. Install Git

Check whether Git is already installed:

```bash
git --version
```

If that prints a version number, go to the next step.

- **Windows:** install Git from [git-scm.com/download/win](https://git-scm.com/download/win). Keep the default options. Close PowerShell and open it again.
- **Mac:** run `xcode-select --install` and click Install.
- **Linux (Ubuntu):** run `sudo apt install git`.

### 5. Download this project

```bash
git clone https://github.com/Marcuss2106/italian_brainrot_image_classifier.git
cd italian_brainrot_image_classifier
```

Stay in this folder for the rest of the commands.

### 6. Install uv

uv downloads Python for you.

Mac or Linux:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Windows PowerShell:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Close the terminal, open a new one, then run:

```bash
cd italian_brainrot_image_classifier
uv --version
```

### 7. Install the libraries

```bash
uv sync
```

The first run takes a few minutes.

### 8. Open this notebook in your browser

```bash
uv run jupyter notebook notebooks/01_classify_italian_brainrot.ipynb
```

A browser tab opens. Press **Shift+Enter** to run each cell. Press **Ctrl+C** in the terminal when you are finished.

## Download the images

This uses the Kaggle token from setup.

In [ ]:
import os

import kagglehub
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

dataset_path = kagglehub.dataset_download("bubblepw/italian-brainrot-images")
image_folder = os.path.join(dataset_path, "brainrot_dataset")

class_names = [
    "ballerina_cappuccina",
    "bombardino_crocodilo",
    "cappuccino_assassino",
    "tralalero_tralala",
    "tung_tung_sahur",
]


def photos_in(class_name):
    folder = os.path.join(image_folder, class_name)
    photos = []
    for file_name in sorted(os.listdir(folder)):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            photos.append(os.path.join(folder, file_name))
    return photos


for class_name in class_names:
    print(class_name, len(photos_in(class_name)))

## Look at one photo per character

In [ ]:
plt.figure(figsize=(12, 3))
for i, class_name in enumerate(class_names):
    photo = photos_in(class_name)[0]
    plt.subplot(1, 5, i + 1)
    plt.imshow(Image.open(photo))
    plt.title(class_name.replace("_", " "), fontsize=9)
    plt.axis("off")
plt.show()

## Train

Wait for the percent at the end. That is the score on photos the model did not train on.

In [ ]:
model = YOLO("yolo11n-cls.pt")
metrics = model.train(data=image_folder, epochs=5, imgsz=224, plots=False)
print(f"Got {metrics.top1:.0%} of the held-out photos right")

## Pick a photo

Leave `pick` as `"any"`, or type a character name. Run the cell again for a new photo.

In [ ]:
import random

pick = "any"  # or "tralalero_tralala", "tung_tung_sahur", ...

if pick == "any":
    class_name = random.choice(class_names)
else:
    class_name = pick

photo = random.choice(photos_in(class_name))

plt.figure(figsize=(4, 4))
plt.imshow(Image.open(photo))
plt.axis("off")
plt.show()

## See the scores

The tallest bar is the model's answer. The title is the real folder.

In [ ]:
result = model(photo, verbose=False)[0]

labels = []
scores = []
for class_id, confidence in zip(result.probs.top5, result.probs.top5conf):
    labels.append(result.names[class_id].replace("_", " "))
    scores.append(float(confidence))

plt.barh(labels[::-1], scores[::-1])
plt.xlim(0, 1)
plt.title(f"Real answer: {class_name.replace('_', ' ')}")
plt.show()

## Try your own photo

The model only knows the five characters, so it will still pick one of them. Choose a picture, then click the button.

In [ ]:
from io import BytesIO

import ipywidgets as widgets
from IPython.display import clear_output, display

uploader = widgets.FileUpload(accept="image/*", multiple=False)
button = widgets.Button(description="What is this?")
output = widgets.Output()

def show_guess(_):
    with output:
        clear_output()
        if not uploader.value:
            print("Choose a photo first.")
            return
        image = Image.open(BytesIO(uploader.value[0]["content"])).convert("RGB")
        result = model(image, verbose=False)[0]
        labels = []
        scores = []
        for class_id, confidence in zip(result.probs.top5, result.probs.top5conf):
            labels.append(result.names[class_id].replace("_", " "))
            scores.append(float(confidence))
        plt.figure(figsize=(4, 4))
        plt.imshow(image)
        plt.axis("off")
        plt.show()
        plt.barh(labels[::-1], scores[::-1])
        plt.xlim(0, 1)
        plt.show()

button.on_click(show_guess)
display(uploader, button, output)